In [21]:
import torch
import math

In [15]:
# -------------------------------------------------
# 1️⃣ Input embeddings (6 tokens, 3-dim each)
# -------------------------------------------------
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your
        [0.55, 0.87, 0.66],  # journey
        [0.57, 0.85, 0.64],  # starts
        [0.22, 0.58, 0.33],  # with
        [0.77, 0.25, 0.10],  # one
        [0.05, 0.80, 0.55],  # step
    ],
    dtype=torch.float32
)

In [16]:
# -------------------------------------------------
# 2️⃣ Define dimensions
# -------------------------------------------------
d_in = inputs.shape[1]   # 3
d_out = 2                # projection dimension

torch.manual_seed(123)

In [17]:
# -------------------------------------------------
# 3️⃣ Initialize projection matrices
# -------------------------------------------------
W_query = torch.rand(d_in, d_out)
W_key   = torch.rand(d_in, d_out)
W_value = torch.rand(d_in, d_out)

In [18]:
# -------------------------------------------------
# 4️⃣ Compute Q, K, V
# -------------------------------------------------
queries = inputs @ W_query     # (6,2)
keys    = inputs @ W_key       # (6,2)
values  = inputs @ W_value     # (6,2)

In [19]:
# -------------------------------------------------
# 5️⃣ Compute Attention Scores
# -------------------------------------------------
attn_scores = queries @ keys.T   # (6,6)

In [26]:
# -------------------------------------------------
# 6️⃣ Scale
# -------------------------------------------------
scale = math.sqrt(d_out)
attn_scores = attn_scores / scale

In [27]:
# -------------------------------------------------
# 7️⃣ Softmax → Attention Weights
# -------------------------------------------------
attn_weights = torch.softmax(attn_scores, dim=-1)

In [28]:
# -------------------------------------------------
# 8️⃣ Compute Context Vectors
# -------------------------------------------------
context_vectors = attn_weights @ values   # (6,2)

In [29]:
# -------------------------------------------------
# 9️⃣ Print Results
# -------------------------------------------------
print("Queries shape:", queries.shape)
print("Keys shape:", keys.shape)
print("Attention Scores shape:", attn_scores.shape)
print("Attention Weights shape:", attn_weights.shape)
print("Context Vectors shape:", context_vectors.shape)

print("\nContext Vectors:\n", context_vectors)

Queries shape: torch.Size([6, 2])
Keys shape: torch.Size([6, 2])
Attention Scores shape: torch.Size([6, 6])
Attention Weights shape: torch.Size([6, 6])
Context Vectors shape: torch.Size([6, 2])

Context Vectors:
 tensor([[0.2940, 0.7919],
        [0.2988, 0.8038],
        [0.2986, 0.8032],
        [0.2905, 0.7834],
        [0.2890, 0.7799],
        [0.2936, 0.7909]])


In [30]:
import torch
import torch.nn as nn
import math

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        # Learnable projection matrices
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

        self.d_out = d_out

    def forward(self, x):
        """
        x shape: (seq_len, d_in)
        """

        # Project inputs to Q, K, V
        queries = x @ self.W_query     # (seq_len, d_out)
        keys    = x @ self.W_key       # (seq_len, d_out)
        values  = x @ self.W_value     # (seq_len, d_out)

        # Compute attention scores
        attn_scores = queries @ keys.T   # (seq_len, seq_len)

        # Scale
        attn_scores = attn_scores / math.sqrt(self.d_out)

        # Softmax
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # Compute context vectors
        context = attn_weights @ values  # (seq_len, d_out)

        return context, attn_weights

In [31]:
torch.manual_seed(123)
sa_V1 = SelfAttention_v1(d_in,d_out)
print(sa_V1(inputs))

(tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>), tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]],
       grad_fn=<SoftmaxBackward0>))


In [32]:
import torch
import torch.nn as nn
import math

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        # Linear layers for Q, K, V
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.d_out = d_out

    def forward(self, x):
        """
        x shape: (seq_len, d_in)
        or (batch_size, seq_len, d_in)
        """

        # Compute Q, K, V
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # Attention scores
        attn_scores = queries @ keys.transpose(-2, -1)

        # Scale
        attn_scores = attn_scores / math.sqrt(self.d_out)

        # Softmax
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # Context vectors
        context = attn_weights @ values

        return context, attn_weights

In [33]:
torch.manual_seed(123)
sa_V2 = SelfAttention_v1(d_in,d_out)
print(sa_V2(inputs))

(tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>), tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>))
